# Week 2 Day 2 — Advanced Pandas

Generated for Cynaris Solutions — Week 2. Run cells from top to bottom.


## Load the three related datasets
These are separate operational, appointment, and patient-feedback tables.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data" / "week-2" / "raw"
resources = pd.read_csv(DATA / "sunrise_resource_utilisation.csv")
satisfaction = pd.read_csv(DATA / "sunrise_patient_satisfaction.csv")
appointments = pd.read_csv(DATA / "da_healthcare_appointments.csv")
print(resources.shape, satisfaction.shape, appointments.shape)


## 1. Demonstrate all merge join types
Merge the two Sunrise tables on hospital and department. These are not necessarily one-to-one; inspect row counts and validate meaning before interpreting.

In [ ]:
keys = ["hospital", "department"]
for how in ["inner", "left", "right", "outer"]:
    merged = pd.merge(resources, satisfaction, on=keys, how=how, suffixes=("_resource", "_survey"))
    print(f"{how.title()} join:", merged.shape)


## 2. Pivot table — resource utilisation by hospital and department

In [ ]:
pivot = pd.pivot_table(resources, index="hospital", columns="department",
                        values="occupied_beds", aggfunc="mean", fill_value=0)
display(pivot)


## 3. Melt — reshape wide attendance data into long format

In [ ]:
attendance_long = pd.melt(
    resources,
    id_vars=["record_id", "hospital", "department", "date"],
    value_vars=["doctor_attendance_pct", "nurse_attendance_pct"],
    var_name="staff_type", value_name="attendance_pct"
)
display(attendance_long.head(10))


## 4. Method chaining

In [ ]:
department_summary = (
    resources.assign(
        occupancy_pct=100 * resources["occupied_beds"] / resources["total_beds"].replace(0, np.nan)
    )
    .groupby("department", as_index=False)
    .agg(records=("record_id", "count"),
         mean_occupancy_pct=("occupancy_pct", "mean"),
         mean_wait_mins=("avg_patient_wait_mins", "mean"))
    .sort_values("mean_wait_mins", ascending=False)
)
display(department_summary)


## 5. Concatenation versus merge
`merge` combines columns using matching keys; `concat` stacks rows or columns along an axis.

In [ ]:
first_half = resources.iloc[:200]
second_half = resources.iloc[200:]
combined_rows = pd.concat([first_half, second_half], ignore_index=True)
print("Original rows:", len(resources), "Concatenated rows:", len(combined_rows))
